# 02 - Feature Engineering

Ce notebook prépare les données brutes pour l'entraînement des modèles de recommandation.
Il charge les jeux de données MovieLens 100K, MovieLens 1M et RetailRocket, nettoie les interactions, encode les IDs, et extrait les features contextuelles.

L'objectif est de produire des versions prêtes à l'emploi dans `data/processed/`.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str((Path('..') / 'src').resolve()))
from data_utils import load_movielens_100k, load_movielens_1m, load_retailrocket, filter_interactions, encode_ids
from context_features import extract_all_context_features

print('Python:', sys.version.split()[0])
print('pandas :', pd.__version__)
print('numpy :', np.__version__)


Python: 3.12.13
pandas : 3.0.3
numpy : 2.4.6


## Chemins des données

On définit les chemins d'accès pour les données brutes et les données traitées.


In [2]:
NOTEBOOK_ROOT = Path('.')
RAW_ROOT = (NOTEBOOK_ROOT / '..' / 'data' / 'raw').resolve()
PROCESSED_ROOT = (NOTEBOOK_ROOT / '..' / 'data' / 'processed').resolve()
print('RAW_ROOT =', RAW_ROOT)
print('PROCESSED_ROOT =', PROCESSED_ROOT)

processed_paths = {
    'ml100k': PROCESSED_ROOT / 'movielens_100k',
    'ml1m': PROCESSED_ROOT / 'movielens_1m',
    'retailrocket': PROCESSED_ROOT / 'retailrocket'
}
for path in processed_paths.values():
    path.mkdir(parents=True, exist_ok=True)
    print(path)


RAW_ROOT = /home/mrtds/Documents/my_projects/context-aware-recsys/data/raw
PROCESSED_ROOT = /home/mrtds/Documents/my_projects/context-aware-recsys/data/processed
/home/mrtds/Documents/my_projects/context-aware-recsys/data/processed/movielens_100k
/home/mrtds/Documents/my_projects/context-aware-recsys/data/processed/movielens_1m
/home/mrtds/Documents/my_projects/context-aware-recsys/data/processed/retailrocket


## 1. MovieLens 100K

Nous nettoyons et enrichissons le dataset MovieLens 100K.
Nous appliquons un filtrage itératif pour supprimer les utilisateurs et items avec moins de 5 interactions, puis nous extrayons des features temporelles et de session.


In [3]:
ml100k_dir = RAW_ROOT / 'movielens' / 'ml-100k'
ml100k = load_movielens_100k(ml100k_dir)
print('raw ml100k shape:', ml100k.shape)
print(ml100k.head())

ml100k_filtered = filter_interactions(ml100k, min_interactions=5)
print('filtered ml100k shape:', ml100k_filtered.shape)

ml100k_encoded = encode_ids(ml100k_filtered)
ml100k_features = extract_all_context_features(ml100k_encoded)

print('processed ml100k columns:', ml100k_features.columns.tolist())
print(ml100k_features[['user_id_encoded', 'item_id_encoded', 'hour_of_day', 'dow_sin', 'session_length']].head())


raw ml100k shape: (100000, 4)
   user_id  item_id  rating           timestamp
0      196      242       3 1997-12-04 15:55:49
1      186      302       3 1998-04-04 19:22:22
2       22      377       1 1997-11-07 07:18:36
3      244       51       2 1997-11-27 05:02:03
4      166      346       1 1998-02-02 05:33:16
Début du filtrage itératif (min 5 interactions)...
filtered ml100k shape: (99287, 4)
processed ml100k columns: ['user_id', 'item_id', 'rating', 'timestamp', 'user_id_encoded', 'item_id_encoded', 'hour_of_day', 'day_of_week', 'is_weekend', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'time_since_last_interaction', 'session_id', 'session_length', 'session_position', 'session_position_norm', 'time_since_last_interaction_log']
   user_id_encoded  item_id_encoded  hour_of_day  dow_sin  session_length
0                0              166           21      0.0              20
1                0              170           21      0.0              20
2                0              

In [4]:
ml100k_features.to_parquet(processed_paths['ml100k'] / 'movielens_100k_processed.parquet', index=False)
print('Saved MovieLens 100K processed dataset.')


Saved MovieLens 100K processed dataset.


## 2. MovieLens 1M

Le même pipeline s'applique à MovieLens 1M pour obtenir un deuxième dataset d'entraînement.


In [10]:
ml1m_dir = RAW_ROOT / 'movielens' / 'ml-1m'
ml1m = load_movielens_1m(ml1m_dir)
print('raw ml1m shape:', ml1m.shape)
print(ml1m.head())

ml1m_filtered = filter_interactions(ml1m, min_interactions=5)
print('filtered ml1m shape:', ml1m_filtered.shape)

ml1m_encoded = encode_ids(ml1m_filtered)
ml1m_features = extract_all_context_features(ml1m_encoded)

print('processed ml1m columns:', ml1m_features.columns.tolist())
print(ml1m_features[['user_id_encoded', 'item_id_encoded', 'hour_of_day', 'dow_cos', 'time_since_last_interaction_log']].head())


raw ml1m shape: (1000209, 4)
   user_id  item_id  rating           timestamp
0        1     1193       5 2000-12-31 22:12:40
1        1      661       3 2000-12-31 22:35:09
2        1      914       3 2000-12-31 22:32:48
3        1     3408       4 2000-12-31 22:04:35
4        1     2355       5 2001-01-06 23:38:11
Début du filtrage itératif (min 5 interactions)...
filtered ml1m shape: (999611, 4)
processed ml1m columns: ['user_id', 'item_id', 'rating', 'timestamp', 'user_id_encoded', 'item_id_encoded', 'hour_of_day', 'day_of_week', 'is_weekend', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'time_since_last_interaction', 'session_id', 'session_length', 'session_position', 'session_position_norm', 'time_since_last_interaction_log']
   user_id_encoded  item_id_encoded  hour_of_day  dow_cos  \
0                0             2757           22  0.62349   
1                0             1068           22  0.62349   
2                0             1444           22  0.62349   
3             

In [6]:
ml1m_features.to_parquet(processed_paths['ml1m'] / 'movielens_1m_processed.parquet', index=False)
print('Saved MovieLens 1M processed dataset.')


Saved MovieLens 1M processed dataset.


## 3. RetailRocket

RetailRocket utilise des événements implicites (`view`, `addtocart`, `transaction`).
Nous codons ces types d'événements et extrayons les mêmes features de session et temporelles.


In [7]:
rr = load_retailrocket(RAW_ROOT / 'retailrocket')
print('raw retailrocket shape:', rr.shape)
print(rr.head())

event_map = {'view': 0, 'addtocart': 1, 'transaction': 2}
rr['event_type'] = rr['event'].map(event_map).fillna(-1).astype(int)
rr['event_weight'] = rr['event_type'] + 1

rr_filtered = filter_interactions(rr, min_interactions=5)
print('filtered retailrocket shape:', rr_filtered.shape)

rr_encoded = encode_ids(rr_filtered)
rr_features = extract_all_context_features(rr_encoded, is_retail_rocket=True)

print('processed retailrocket columns:', rr_features.columns.tolist())
print(rr_features[['user_id_encoded', 'item_id_encoded', 'event_type', 'session_length', 'is_desktop_proxy']].head())


raw retailrocket shape: (2756101, 5)
                timestamp  user_id event  item_id  transactionid
0 2015-06-02 05:02:12.117   257597  view   355908            NaN
1 2015-06-02 05:50:14.164   992329  view   248676            NaN
2 2015-06-02 05:13:19.827   111016  view   318965            NaN
3 2015-06-02 05:12:35.914   483717  view   253185            NaN
4 2015-06-02 05:02:17.106   951259  view   367447            NaN
Début du filtrage itératif (min 5 interactions)...
filtered retailrocket shape: (781519, 7)
processed retailrocket columns: ['timestamp', 'user_id', 'event', 'item_id', 'transactionid', 'event_type', 'event_weight', 'user_id_encoded', 'item_id_encoded', 'hour_of_day', 'day_of_week', 'is_weekend', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'time_since_last_interaction', 'session_id', 'session_length', 'session_position', 'session_position_norm', 'time_since_last_interaction_log', 'is_desktop_proxy']
   user_id_encoded  item_id_encoded  event_type  session_length  \

In [8]:
rr_features.to_parquet(processed_paths['retailrocket'] / 'retailrocket_processed.parquet', index=False)
print('Saved RetailRocket processed dataset.')


Saved RetailRocket processed dataset.


## Vérification finale

On vérifie que les fichiers de sortie sont bien présents et que les colonnes clés sont disponibles.


In [9]:
for name, path in processed_paths.items():
    files = sorted(path.glob('*.parquet'))
    print(name, '->', files)


ml100k -> [PosixPath('/home/mrtds/Documents/my_projects/context-aware-recsys/data/processed/movielens_100k/interactions.parquet'), PosixPath('/home/mrtds/Documents/my_projects/context-aware-recsys/data/processed/movielens_100k/movielens_100k_processed.parquet')]
ml1m -> [PosixPath('/home/mrtds/Documents/my_projects/context-aware-recsys/data/processed/movielens_1m/interactions.parquet'), PosixPath('/home/mrtds/Documents/my_projects/context-aware-recsys/data/processed/movielens_1m/movielens_1m_processed.parquet')]
retailrocket -> [PosixPath('/home/mrtds/Documents/my_projects/context-aware-recsys/data/processed/retailrocket/interactions.parquet'), PosixPath('/home/mrtds/Documents/my_projects/context-aware-recsys/data/processed/retailrocket/retailrocket_processed.parquet')]


## Notes

- Les datasets traités sont stockés dans `data/processed/` pour être réutilisés par les notebooks d'entraînement.
- Les IDs utilisateur et item sont encodés en entiers contigus (`user_id_encoded`, `item_id_encoded`).
- Les features temporelles et de session sont extraites pour enrichir les modèles contextuels.
